# Домашнее задание 2: смотрим на Землю из космоса

Спутники Sentinel-2 Европейского космического агентства фотографируют всю сушу Земли примерно раз в пять дней. Снимков столько, что людей на их просмотр никогда не хватит. Поэтому их разбирают нейросети: следят за вырубкой лесов, считают поля, замечают, как растут города и где разлились реки.

В этой домашке вы соберёте такую сеть сами. Датасет называется EuroSAT: 27 000 снимков 64 на 64 пикселя с территории Европы и 10 типов местности (лес, река, шоссе, жилой район, промзона и так далее). Один пиксель соответствует 10 метрам, так что каждая картинка показывает квадрат примерно 640 на 640 метров.

План такой:
1. собрать свёрточную сеть с нуля и дотянуть её до accuracy 0.85 на тесте;
2. написать residual-блок своими руками и собрать из таких блоков маленький ResNet. Потом дообучить настоящую ResNet-18, которую до нас обучили на миллионе фотографий из ImageNet;
3. разобраться, где сеть ошибается и почему;
4. заглянуть внутрь сети: что видят свёрточные слои и куда сеть смотрит, когда принимает решение. А в конце, если захочется, проверить её на спутниковом снимке своего района.

Оцениваются только части 1 и 2. В частях 3 и 4 весь код уже написан, их нужно просто запустить и прочитать. Там самое интересное: мы посмотрим на обученные вами модели изнутри.

Обучать свёрточные сети на процессоре можно, но долго. Удобнее запускать ноутбук в Google Colab с GPU (Среда выполнения > Сменить среду выполнения > T4 GPU). Код ниже сам переходит на видеокарту, если она есть.

Код пишете там, где стоит `# ваш код`. Остальные ячейки просто запускаете. На вопросы отвечайте в ячейках «Ответы» своими словами, хватит 2-3 предложений на вопрос.

### Оценивание

| Часть | Что смотрим | Баллы |
|---|---|---|
| 1. CNN с нуля | Dataset и аугментации, модель `SatNet`, функции `train_model` и `predict`, accuracy на тесте не ниже 0.85, ответы на вопросы | 4.5 |
| 2. ResNet | `ResidualBlock`, `MiniResNet` и её обучение , дообучение ResNet-18 в двух вариантах, сравнение моделей и ответы на вопросы | 5.5 |
| 3. Разбор ошибок | не оценивается, только запустить и прочитать | |
| 4. Внутри сети | не оценивается, только запустить и прочитать | |

Перед отправкой сделайте `Kernel > Restart & Run All` и проверьте, что весь ноутбук, включая части 3 и 4, выполняется без ошибок. Части 3 и 4 используют ваши модели, так что заодно это хорошая проверка частей 1 и 2. Если ячейка падает, задание из неё не засчитывается. Сдаёте `.ipynb` с выводами и графиками.

In [ ]:
import os
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
from torchvision import datasets, models

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

## Данные

Ячейка ниже скачивает EuroSAT (около 90 Мб), делит его на train, val и test и раскладывает в формате, который мы использовали на лекции: в каждой папке лежат картинки и `format.json`, где имени файла сопоставлен номер класса.

Если скачивание оборвалось, удалите папки `eurosat` и `eurosat_raw` и запустите ячейку ещё раз. Если torchvision ругается на адрес загрузки, обновите его: `pip install -U torchvision`.

In [ ]:
CLASSES = ["AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
           "Pasture", "PermanentCrop", "Residential", "River", "SeaLake"]
CLASSES_RU = ["пашня", "лес", "луг", "шоссе", "промзона",
              "пастбище", "сады", "жилой район", "река", "водоём"]


def make_dataset(path="eurosat", seed=42):
    if os.path.exists(path):
        print("Датасет уже готов")
        return

    raw = datasets.EuroSAT(root="eurosat_raw", download=True)
    assert raw.classes == CLASSES

    samples = list(raw.samples)
    random.Random(seed).shuffle(samples)
    n = len(samples)
    parts = {
        "train": samples[:int(0.7 * n)],
        "val": samples[int(0.7 * n):int(0.85 * n)],
        "test": samples[int(0.85 * n):],
    }

    for split, items in parts.items():
        folder = os.path.join(path, split)
        os.makedirs(folder)
        fmt = {}
        for src, target in tqdm(items, desc=split):
            name = os.path.basename(src)
            shutil.copy(src, os.path.join(folder, name))
            fmt[name] = target
        with open(os.path.join(folder, "format.json"), "w") as fp:
            json.dump(fmt, fp)

    shutil.rmtree("eurosat_raw")


make_dataset()

In [ ]:
with open("eurosat/train/format.json") as fp:
    fmt = json.load(fp)

fig, axes = plt.subplots(4, 10, figsize=(16, 7))
for j, name in enumerate(CLASSES_RU):
    files = [f for f, t in fmt.items() if t == j][:4]
    for i, f in enumerate(files):
        axes[i, j].imshow(Image.open(os.path.join("eurosat/train", f)))
        axes[i, j].axis("off")
    axes[0, j].set_title(name)
plt.show()

---
## Часть 1. Свёрточная сеть с нуля (4.5 балла)

### 1.1 Dataset

Напишите `SatDataset` по образцу `SunDataset` с лекции. Разница две:
- вместо `train=True/False` будет параметр `split`, который принимает значения `"train"`, `"val"` или `"test"`. Это имя папки;
- в `SunDataset` решалась задача регрессии, поэтому метка была `float32`. У нас классификация, и `nn.CrossEntropyLoss` ждёт номер класса типа `torch.long`.

In [ ]:
class SatDataset(data.Dataset):
    def __init__(self, path, split="train", transform=None):
        pass  # ваш код

    def __getitem__(self, item):
        pass  # ваш код

    def __len__(self):
        pass  # ваш код

### 1.2 Аугментации

У спутникового снимка нет верха и низа: лес, повёрнутый вверх ногами, остаётся лесом. Значит, во время обучения картинки можно случайно отражать, и сеть будет видеть больше разных примеров.

`transforms` ниже такой же, как на лекции. Напишите `train_transforms`: то же самое, но перед `ToDtype` добавьте случайное горизонтальное и вертикальное отражение (`tfs.RandomHorizontalFlip()` и `tfs.RandomVerticalFlip()`).

In [ ]:
transforms = tfs.Compose([tfs.ToImage(), tfs.ToDtype(torch.float32, scale=True)])

train_transforms = ...  # ваш код

### 1.3 DataLoader

Создайте три датасета и три загрузчика с `batch_size=64`:
- `d_train` и `train_data` с аугментациями и перемешиванием;
- `d_val` и `val_data` без аугментаций и без перемешивания;
- `d_test` и `test_data` так же, как val.

In [ ]:
# ваш код

In [ ]:
# проверка
x, y = next(iter(train_data))
print(x.shape, x.dtype)  # torch.Size([64, 3, 64, 64]) torch.float32
print(y.shape, y.dtype)  # torch.Size([64]) torch.int64
print(len(d_train), len(d_val), len(d_test))

Дальше несколько готовых функций для картинок и графиков, они понадобятся во всех частях. Просто запустите ячейку.

In [ ]:
def show_images(dataset, indices, titles=None, colors=None, ncols=8):
    "Показывает картинки dataset[i] для i из indices, с подписями и цветом подписей."
    nrows = (len(indices) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(2 * ncols, 2.4 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for k, idx in enumerate(indices):
        img, _ = dataset[int(idx)]
        ax = axes.flat[k]
        ax.imshow(img.permute(1, 2, 0).clamp(0, 1))
        if titles is not None:
            ax.set_title(titles[k], fontsize=9, color=colors[k] if colors else "black")
    plt.tight_layout()
    plt.show()


def plot_history(histories):
    "histories: словарь {название модели: (loss_lst, loss_lst_val, acc_lst_val)}"
    fig, axes = plt.subplots(1, 3, figsize=(17, 4))
    for name, (loss_lst, loss_lst_val, acc_lst_val) in histories.items():
        epochs_range = range(1, len(loss_lst) + 1)
        axes[0].plot(epochs_range, loss_lst, marker="o", label=name)
        axes[1].plot(epochs_range, loss_lst_val, marker="o", label=name)
        axes[2].plot(epochs_range, acc_lst_val, marker="o", label=name)
    for ax, title in zip(axes, ["loss на train", "loss на validation", "accuracy на validation"]):
        ax.set_title(title)
        ax.set_xlabel("эпоха")
        ax.grid(alpha=0.3)
        ax.legend()
    plt.show()


def plot_confusion(cm, class_names, hide_diagonal=False):
    "Рисует матрицу ошибок. hide_diagonal=True прячет диагональ, чтобы лучше были видны ошибки."
    cm = np.array(cm, dtype=float)
    if hide_diagonal:
        np.fill_diagonal(cm, 0)
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(class_names)), class_names, rotation=45, ha="right")
    ax.set_yticks(range(len(class_names)), class_names)
    ax.set_xlabel("ответ сети")
    ax.set_ylabel("правильный ответ")
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, int(cm[i, j]), ha="center", va="center", color=color, fontsize=8)
    plt.tight_layout()
    plt.show()

Посмотрим, как работают аугментации: одна и та же картинка, пропущенная через `train_transforms` восемь раз.

In [ ]:
show_images(d_train, [0] * 8)

### 1.4 Модель

Соберите `SatNet` по схеме:

| Слой | Что делает | Размер на выходе |
|---|---|---|
| вход | | 3 × 64 × 64 |
| `conv1` | Conv2d(3, 32, kernel_size=3, padding=1), ReLU, MaxPool2d(2) | 32 × 32 × 32 |
| `conv2` | Conv2d(32, 64, kernel_size=3, padding=1), ReLU, MaxPool2d(2) | ? |
| `conv3` | Conv2d(64, 128, kernel_size=3, padding=1), ReLU, MaxPool2d(2) | ? |
| | flatten | ? |
| `fc1` | Linear(?, 128), ReLU | 128 |
| `fc2` | Linear(128, 10) | 10 |

Знаки вопроса заполните сами. Атрибуты назовите именно `conv1`, `conv2`, `conv3`, `fc1`, `fc2`: в части 4 мы будем доставать `conv1` и `conv2`, чтобы посмотреть, что они видят.

In [ ]:
class SatNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # ваш код

    def forward(self, x):
        pass  # ваш код

In [ ]:
# проверка
model_cnn = SatNet()
print(model_cnn(x).shape)  # torch.Size([64, 10])
print("параметров:", sum(p.numel() for p in model_cnn.parameters()))

### 1.5 Обучение

Напишите две функции.

`train_model` можно почти целиком взять из прошлой домашки. Она возвращает `loss_lst`, `loss_lst_val` и `acc_lst_val`. Новое здесь одно: батчи нужно переносить на `device` (`x.to(device)`, `y.to(device)`), иначе на GPU ничего не заработает. И метки теперь номера классов, а не one-hot, так что для accuracy `argmax` от `y` брать не нужно.

`predict` прогоняет модель по загрузчику и возвращает два тензора на CPU:
- `probs` размером (N, 10): вероятности классов (не забудьте softmax);
- `targets` размером (N,): правильные ответы.

Из `predict` accuracy считается в одну строку, а в части 3 нам понадобятся сами вероятности.

In [ ]:
def train_model(model, optimizer, train_data, val_data, epochs, verbose=True):
    loss_function = nn.CrossEntropyLoss()
    loss_lst = []
    loss_lst_val = []
    acc_lst_val = []

    # ваш код

    return loss_lst, loss_lst_val, acc_lst_val


def predict(model, loader):
    pass  # ваш код

Обучите `SatNet` с Adam (`lr=0.001`) 10 эпох, нарисуйте кривые через `plot_history` и посчитайте accuracy на тесте. **Нужно получить не меньше 0.85.**

<details>
<summary>Если не получается дотянуть до 0.85</summary>

<br>

- проверьте, что `train_data` действительно с `shuffle=True`, а картинки нормированы в [0, 1];
- обучите подольше, например 15 эпох;
- добавьте `nn.BatchNorm2d` после каждой свёртки и Dropout перед `fc2`. Всё это у вас уже было в первой домашке.

</details>

In [ ]:
torch.manual_seed(42)

# ваш код

**Вопросы**
1. Почему для спутниковых снимков можно отражать картинку вверх ногами, а, например, для фотографий котов или для рукописных цифр так делать не стоит?
2. Какие размеры у вас получились на месте знаков вопроса в таблице? В каком слое `SatNet` больше всего параметров и почему?

**Ответы:**

1.

2.

---
## Часть 2. ResNet (5.5 баллов)

Напомню идею с лекции. Обычный блок сети учит преобразование $F(x)$ и отдаёт его дальше. Residual-блок отдаёт $F(x) + x$: вход проходит в обход слоёв по короткому пути (skip connection) и прибавляется к результату. Слоям остаётся выучить только поправку к входу, а градиент по короткому пути без потерь доходит до первых слоёв. Благодаря этому получается обучать сети в десятки и сотни слоёв.

### 2.1 Residual-блок

Напишите `ResidualBlock(channels)`. Число каналов на входе и выходе одинаковое, размер картинки не меняется:

```
вход x
  conv1: Conv2d(channels, channels, 3, padding=1), затем bn1: BatchNorm2d, затем ReLU
  conv2: Conv2d(channels, channels, 3, padding=1), затем bn2: BatchNorm2d
  прибавить x
  ReLU
```

Обратите внимание: последний ReLU стоит после сложения.

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        # ваш код

    def forward(self, x):
        pass  # ваш код

Хитрая проверка. Если обнулить веса `bn2`, то основная ветка блока всегда будет выдавать ноль, и от блока останется только короткий путь: $\mathrm{ReLU}(0 + x) = \mathrm{ReLU}(x)$. Если skip connection сделан правильно, проверка напечатает `True`.

In [ ]:
block = ResidualBlock(16)
x_test = torch.randn(4, 16, 32, 32)
print(block(x_test).shape)  # torch.Size([4, 16, 32, 32])

with torch.no_grad():
    block.bn2.weight.zero_()
print(torch.allclose(block(x_test), F.relu(x_test)))

### 2.2 MiniResNet

Соберите маленький ResNet из трёх стадий. Каждая стадия устроена так: свёртка, которая меняет число каналов, потом BatchNorm и ReLU, потом residual-блок, потом MaxPool2d(2).

| Стадия | Что внутри | Размер на выходе |
|---|---|---|
| 1 | `conv1`: Conv2d(3, 32, 3, padding=1), `bn1`, ReLU, `block1`: ResidualBlock(32), MaxPool2d(2) | 32 × 32 × 32 |
| 2 | `conv2`: Conv2d(32, 64, 3, padding=1), `bn2`, ReLU, `block2`: ResidualBlock(64), MaxPool2d(2) | 64 × 16 × 16 |
| 3 | `conv3`: Conv2d(64, 128, 3, padding=1), `bn3`, ReLU, `block3`: ResidualBlock(128), MaxPool2d(2) | 128 × 8 × 8 |
| выход | `nn.AdaptiveAvgPool2d(1)`, flatten, `fc`: Linear(128, 10) | 10 |

В конце вместо большого полносвязного слоя стоит global average pooling: каждая из 128 карт признаков усредняется в одно число. Так сделано и в настоящих ResNet.

Обучите `MiniResNet` так же, как `SatNet`: Adam, `lr=0.001`, 10 эпох, `torch.manual_seed(42)` перед созданием модели. Нарисуйте кривые обеих моделей на одних графиках и выведите для обеих число параметров и accuracy на тесте.

In [ ]:
class MiniResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # ваш код

    def forward(self, x):
        pass  # ваш код

In [ ]:
torch.manual_seed(42)

model_res = MiniResNet().to(device)
# ваш код

<details>
<summary><b>Пояснение: почему MiniResNet может обогнать SatNet совсем ненамного</b> (откройте после того, как посмотрите на результаты)</summary>

<br>

Skip connections придумали, чтобы решить проблему очень глубоких сетей. Когда слоёв 50 или 100, градиенту при обратном проходе нужно пройти через каждый из них, и по дороге он затухает или, наоборот, раздувается. В статье, где впервые предложили ResNet, есть показательный пример: обычная сеть из 56 слоёв обучалась хуже, чем сеть из 20 слоёв, причём даже на train. То есть дело было не в переобучении, а в том, что глубокую сеть просто не получалось нормально обучить. Короткий путь в обход слоёв это исправляет: градиент доходит по нему до первых слоёв почти без изменений.

В `SatNet` всего 3 свёрточных слоя, в `MiniResNet` их 9. На такой глубине градиент нормально проходит и без skip connections, поэтому большого выигрыша от них ждать не стоит. Если `MiniResNet` всё же заметно лучше, скорее всего, дело в двух других вещах: в BatchNorm, который стабилизирует обучение, и в global average pooling. Посмотрите на число параметров: в `SatNet` больше миллиона параметров, и почти все они в `fc1`, а в `MiniResNet` их в несколько раз меньше. Меньше параметров, меньше риск переобучиться.

Настоящая сила ResNet проявляется там, где слоёв много. Именно такую сеть мы сейчас и возьмём.

</details>

### 2.3 Дообучение ResNet-18

ResNet-18 это 18 слоёв и около 11 миллионов параметров. Обучать такую сеть с нуля на 19 тысячах картинок долго и почти бессмысленно. Но в `torchvision` есть версия, которую уже обучили на ImageNet: это 1.2 миллиона фотографий, 1000 классов, от собак до грузовиков. Её мы и дообучим на спутниковых снимках.

Сначала данные. Предобученная сеть привыкла к картинкам, нормированным определённым образом, поэтому к трансформациям нужно добавить в конец
`tfs.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])`.
Это среднее и стандартное отклонение по каналам на ImageNet.

Создайте `transforms_in` и `train_transforms_in` (такие же, как раньше, плюс Normalize), три датасета и загрузчики `train_data_in`, `val_data_in`, `test_data_in`.

In [ ]:
# ваш код

Теперь два варианта дообучения, по 3 эпохи каждый.

**Вариант А: заморозить всё, кроме последнего слоя.** Загрузите сеть через `models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)`, выключите градиенты у всех параметров (`p.requires_grad = False`), а затем замените последний слой `model.fc` на новый `nn.Linear(512, 10)`. У нового слоя градиенты включены по умолчанию. Оптимизатор: Adam по `model.fc.parameters()` с `lr=0.001`.

**Вариант Б: дообучить всю сеть.** Снова загрузите сеть, замените `fc`, но ничего не замораживайте. Оптимизатор: Adam по всем параметрам с маленьким `lr=0.0001`, чтобы не сломать то, чему сеть уже научилась.

Для каждого варианта выведите, сколько параметров обучается (`sum(p.numel() for p in model.parameters() if p.requires_grad)`), и сохраните историю обучения. Модели назовите `model_frozen` и `model_finetune`.

На CPU одна эпоха ResNet-18 может идти несколько минут, на GPU около полуминуты.

In [ ]:
torch.manual_seed(42)

# вариант А, ваш код

hist_frozen = train_model(model_frozen, optimizer, train_data_in, val_data_in, epochs=3)

In [ ]:
torch.manual_seed(42)

# вариант Б, ваш код

hist_finetune = train_model(model_finetune, optimizer, train_data_in, val_data_in, epochs=3)

Сведем результаты всех четырёх моделей: кривые обучения через `plot_history` и accuracy на тесте для каждой.

In [ ]:
plot_history({
    "SatNet": hist_cnn,
    "MiniResNet": hist_res,
    "ResNet-18, только fc": hist_frozen,
    "ResNet-18, вся сеть": hist_finetune,
})

for name, m, loader in [("SatNet", model_cnn, test_data),
                        ("MiniResNet", model_res, test_data),
                        ("ResNet-18, только fc", model_frozen, test_data_in),
                        ("ResNet-18, вся сеть", model_finetune, test_data_in)]:
    probs, targets = predict(m, loader)
    acc = (probs.argmax(dim=1) == targets).float().mean().item()
    print(f"{name}: test accuracy {acc:.3f}")

<details>
<summary><b>Пояснение: почему сеть, обученная на фотографиях собак и машин, так хорошо понимает спутниковые снимки</b> (откройте после того, как посмотрите на результаты)</summary>

<br>

Вспомните, что мы говорили про свёрточные сети на лекции: первые слои учатся находить простые вещи вроде границ, цветовых пятен и штриховки. Следующие слои собирают из них текстуры и узоры, и только последние слои отвечают за то, что это именно собака или грузовик.

Границы, текстуры и узоры есть на любых картинках. Ровные ряды деревьев в саду, сетка улиц жилого района, гладкая поверхность воды это всё тоже текстуры. Поэтому почти все слои сети, обученной на ImageNet, полезны и для спутниковых снимков. Новому приходится учить только последний слой. А ImageNet больше нашего датасета в 60 с лишним раз, так что фильтры у предобученной сети получились гораздо лучше тех, что мы можем обучить с нуля.

**Вариант А** использует предобученную сеть как готовый «извлекатель признаков». Каждая картинка превращается в 512 чисел, и сверху обучается обычный линейный классификатор. Это быстро и почти не переобучается: учится всего 5 тысяч параметров.

**Вариант Б** разрешает подстроить под спутниковые снимки все слои, поэтому обычно работает лучше. Learning rate берут маленьким, чтобы сеть аккуратно подправила веса, а не забыла всё, чему научилась на ImageNet.

И ещё одна деталь. ResNet-18 рассчитана на картинки 224 на 224, а наши в 3.5 раза меньше. После всех уменьшений от картинки к концу сети остаётся карта 2 на 2. Сеть всё равно справляется, но если увеличить снимки (например, добавить `tfs.Resize(128)`), accuracy обычно ещё подрастёт. Правда, и обучение станет заметно медленнее.

</details>

**Вопросы**
1. Какая модель оказалась лучшей и на сколько она обогнала `SatNet`? Сравните модели по числу параметров и по времени обучения.
2. В варианте А обучается всего несколько тысяч параметров из 11 миллионов. Почему этого хватает для хорошего результата?

**Ответы:**

1.

2.

---
## Часть 3. Где и почему сеть ошибается

**Дальше ничего писать не нужно, эти части не оцениваются.** Запускайте ячейки по порядку и читайте. Код уже готов, но его стоит хотя бы пролистать: такие же приёмы пригодятся вам в любом проекте.

Accuracy 95% звучит отлично, но одно число мало что говорит о модели. Где именно она ошибается? Ошибки случайные или в них есть система? Может быть, иногда ошибается не сеть, а разметка? Чтобы ответить на такие вопросы, на ошибки нужно посмотреть глазами. Этим и займёмся.

Сначала выберем лучшую из четырёх моделей, которые вы обучили, и дальше будем работать с ней.

In [ ]:
candidates = [
    ("SatNet", model_cnn, test_data, transforms),
    ("MiniResNet", model_res, test_data, transforms),
    ("ResNet-18, только fc", model_frozen, test_data_in, transforms_in),
    ("ResNet-18, вся сеть", model_finetune, test_data_in, transforms_in),
]

best_acc = 0
for name, m, loader, tr in candidates:
    probs_m, targets_m = predict(m, loader)
    acc = (probs_m.argmax(dim=1) == targets_m).float().mean().item()
    print(f"{name}: test accuracy {acc:.3f}")
    if acc > best_acc:
        best_acc, best_name = acc, name
        best_model, best_test_data, best_transform = m, loader, tr
        probs, targets = probs_m, targets_m

best_test_ds = best_test_data.dataset
preds = probs.argmax(dim=1)
conf = probs.max(dim=1).values
print("\nДальше работаем с моделью:", best_name)

### 3.1 Accuracy по каждому классу

Общая accuracy усредняет всё подряд. Модель может почти идеально узнавать воду и лес и при этом регулярно путать пастбища с лугами, а в среднем всё равно получится красивое число. Поэтому посмотрим на каждый класс отдельно: какая доля снимков этого класса распознана верно.

In [ ]:
class_acc = [(preds[targets == c] == c).float().mean().item() for c in range(10)]
order = np.argsort(class_acc)

plt.figure(figsize=(7, 4.5))
bars = plt.barh([CLASSES_RU[i] for i in order], [class_acc[i] for i in order], color="tab:blue")
for bar, i in zip(bars, order):
    plt.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height() / 2,
             f"{class_acc[i]:.1%}", va="center", fontsize=9)
plt.xlim(min(class_acc) - 0.05, 1.03)
plt.title("Доля верных ответов по классам")
plt.grid(alpha=0.3, axis="x")
plt.show()

Скорее всего, внизу списка у вас оказались луг, пастбище, сады или шоссе. Это классы, которые и человеку трудно различить на квадрате 640 на 640 метров: луг, пастбище и сады с высоты выглядят как зелёные пятна с похожей текстурой. А наверху почти наверняка лес и водоём: у них очень характерный цвет, и спутать их почти не с чем.

### 3.2 Матрица ошибок

Матрица ошибок (confusion matrix) это таблица 10 на 10. В строке «река» и столбце «шоссе» стоит число рек, которые сеть приняла за шоссе. На диагонали правильные ответы, всё остальное ошибки.

Нарисуем её два раза. На первой картинке диагональ настолько больше всего остального, что ошибок почти не видно. Поэтому на второй мы диагональ спрячем и оставим только ошибки.

In [ ]:
cm = torch.zeros(10, 10, dtype=torch.long)
for t, p in zip(targets, preds):
    cm[t, p] += 1

plot_confusion(cm, CLASSES_RU)
plot_confusion(cm, CLASSES_RU, hide_diagonal=True)

Посмотрите на вторую картинку: ошибки не размазаны по всей таблице, а собраны в нескольких клетках. Значит, сеть путает не что попало, а конкретные пары похожих классов. Это хорошая новость. Систематические ошибки можно понять и исправить, например собрать больше примеров именно этих классов или показать сети снимки покрупнее.

### 3.3 Какие классы путаются чаще всего

Возьмём три самые «горячие» клетки вне диагонали и посмотрим на сами снимки.

In [ ]:
off_diag = cm.clone()
off_diag.fill_diagonal_(0)
top_pairs = torch.argsort(off_diag.flatten(), descending=True)[:3]

for k in top_pairs:
    t, p = divmod(k.item(), 10)
    print(f"Правильно «{CLASSES_RU[t]}», а сеть сказала «{CLASSES_RU[p]}»: {cm[t, p].item()} раз")
    idx = torch.where((targets == t) & (preds == p))[0][:8].tolist()
    titles = [f"сеть: {CLASSES_RU[p]} {conf[i].item():.0%}" for i in idx]
    show_images(d_test, idx, titles)

Обычно эти ошибки легко объяснить, если посмотреть на снимки:
- **шоссе и река** выглядят почти одинаково: длинная полоса через весь кадр, по краям поля. Только у реки полоса тёмная, а у шоссе светлая. А в тени, в мутной воде или зимой эта разница пропадает;
- **луг, пастбище и сады** это зелёные и бурые пятна. Сады отличаются ровными рядами деревьев, но на разрешении 10 метров на пиксель ряды часто просто не видны;
- **пашня и сады** тоже похожи: и там и там прямоугольные участки с полосками.

Человек на таких снимках ошибался бы в тех же местах.

### 3.4 Самые уверенные ошибки

Отдельно интересны ошибки, в которых сеть была уверена на 90% и больше. Посмотрим на восемь самых уверенных.

In [ ]:
wrong = torch.where(preds != targets)[0]
worst = wrong[torch.argsort(conf[wrong], descending=True)[:8]].tolist()

titles = [f"разметка: {CLASSES_RU[targets[i].item()]}\nсеть: {CLASSES_RU[preds[i].item()]} {conf[i].item():.0%}"
          for i in worst]
show_images(d_test, worst, titles, colors=["red"] * len(worst))

Присмотритесь к этим снимкам. Довольно часто оказывается, что сеть не так уж неправа:
- на снимке есть сразу два класса, например река и идущая вдоль неё дорога, или жилой район рядом с промзоной. А метка у снимка одна, и тот, кто размечал, выбрал одну из них;
- иногда разметка просто ошибочна. В любом большом датасете есть такие ошибки, и уверенные ошибки модели это самый быстрый способ их найти.

В реальных проектах так и делают: обучают модель, сортируют ошибки по уверенности и вручную проверяют верх списка. Часто так находится заметная доля ошибок разметки.

### 3.5 Карта: как сеть видит снимки

Перед последним линейным слоем каждая картинка превращается в вектор: у ResNet-18 это 512 чисел, у `SatNet` 128. Это и есть «описание» снимка, которое сеть составила сама. Последний слой просто проводит в этом пространстве границы между классами.

512 измерений увидеть нельзя, но есть алгоритм t-SNE. Он раскладывает точки на плоскости так, чтобы те, что были близки в исходном пространстве, остались рядом и на картинке. Возьмём векторы всех тестовых снимков и посмотрим на такую карту. Каждая точка это один снимок, цвет это правильный класс, чёрные крестики это ошибки сети.

Ячейка считается около минуты.

In [ ]:
from sklearn.manifold import TSNE

# вектор перед последним линейным слоем достаём через hook: функцию, которая
# вызывается при каждом проходе через слой и может подсмотреть его вход
last_linear = [m for m in best_model.modules() if isinstance(m, nn.Linear)][-1]
features = []
hook = last_linear.register_forward_hook(lambda module, inp, out: features.append(inp[0].detach().cpu()))
predict(best_model, best_test_data)
hook.remove()
features = torch.cat(features).numpy()
print("размер векторов:", features.shape)

emb = TSNE(n_components=2, init="pca", random_state=42).fit_transform(features)

plt.figure(figsize=(10, 8))
cmap = plt.get_cmap("tab10")
for c in range(10):
    mask = (targets == c).numpy()
    plt.scatter(emb[mask, 0], emb[mask, 1], s=6, color=cmap(c), label=CLASSES_RU[c])
mistakes = (preds != targets).numpy()
plt.scatter(emb[mistakes, 0], emb[mistakes, 1], s=25, color="black", marker="x", label="ошибки")
plt.legend(markerscale=3, loc="center left", bbox_to_anchor=(1, 0.5))
plt.axis("off")
plt.title("Тестовые снимки глазами сети (t-SNE)")
plt.show()

Каждый класс собрался в своё облако, хотя сети никто не объяснял, как раскладывать снимки по пространству. Она сама научилась описывать их так, чтобы снимки одного класса оказывались рядом.

На карте видно и всё, о чём мы говорили выше:
- облака, которые соприкасаются, это как раз те классы, которые сеть путает. Найдите на карте луг, пастбище и сады;
- большинство крестиков лежит на границах облаков, там, где сеть сомневалась;
- крестики, которые лежат глубоко внутри чужого облака, это главные кандидаты на ошибки разметки. Сеть видит снимок как типичную реку, а метка у него «шоссе».

Одно предупреждение: расстояния между облаками на картинке t-SNE ничего не значат. Алгоритм сохраняет только соседство близких точек. Поэтому то, что лес на картинке далеко от реки, ещё не значит, что сеть считает их очень разными.

---
## Часть 4. Заглядываем внутрь сети

Здесь тоже ничего писать не нужно. Мы посмотрим, что происходит внутри свёрточных слоёв и на какие части снимка сеть опирается, когда принимает решение.

### 4.1 Сеть за работой

Для начала просто 16 случайных снимков из теста. Зелёная подпись означает, что сеть ответила верно, красная, что ошиблась. Рядом с ответом уверенность сети.

In [ ]:
idx = random.Random(0).sample(range(len(d_test)), 16)
titles = [f"{CLASSES_RU[preds[i].item()]} {conf[i].item():.0%}" for i in idx]
colors = ["green" if preds[i] == targets[i] else "red" for i in idx]
show_images(d_test, idx, titles, colors)

### 4.2 Чему научился первый слой ResNet-18

Свёрточный фильтр это маленький шаблон, который скользит по картинке и откликается там, где картинка на него похожа. В первом слое ResNet-18 64 фильтра размером 7 на 7 пикселей, по одному на каждый из трёх цветовых каналов. Их можно нарисовать как маленькие цветные картинки. Возьмём их из сети, предобученной на ImageNet.

In [ ]:
w = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1).conv1.weight.detach()  # (64, 3, 7, 7)

fig, axes = plt.subplots(4, 16, figsize=(16, 4.4))
for k, ax in enumerate(axes.flat):
    f = w[k]
    f = (f - f.min()) / (f.max() - f.min())  # растягиваем значения в [0, 1], чтобы нарисовать как картинку
    ax.imshow(f.permute(1, 2, 0), interpolation="nearest")
    ax.axis("off")
plt.suptitle("64 фильтра первого слоя ResNet-18")
plt.show()

Почти все фильтры это либо перепады яркости под разными углами (детекторы границ и полосок), либо цветные пятна и переходы между цветами. Никто не закладывал их в сеть вручную: она сама пришла к ним, пока училась различать собак, машины и чашки.

Интересно, что живое зрение устроено похоже. В 1959-1962 годах Дэвид Хьюбел и Торстен Визел обнаружили в первичной зрительной коре кошек и обезьян нейроны, которые реагируют на полоски определённого наклона. За это открытие они получили Нобелевскую премию. Первый слой свёрточной сети, обученной на фотографиях, раз за разом приходит к очень похожим детекторам.

Это и объясняет результат части 2. Границы, полоски и цветовые пятна есть на любых картинках, поэтому первые слои сети, обученной на ImageNet, одинаково полезны и для собак, и для снимков из космоса.

### 4.3 Карты признаков

Если приложить один фильтр ко всем местам картинки, получится карта признаков: картинка, которая показывает, где на снимке нашёлся этот шаблон. Посмотрим на карты первых двух слоёв нашей `SatNet` для реки, шоссе и жилого района.

В каждом ряду слева исходный снимок, дальше 16 каналов. Яркие места означают, что фильтр сработал сильно.

In [ ]:
def show_feature_maps(img, n=16):
    model_cnn.eval()
    with torch.no_grad():
        x = img.unsqueeze(0).to(device)
        maps1 = F.relu(model_cnn.conv1(x))
        maps2 = F.relu(model_cnn.conv2(model_cnn.pool(maps1)))

    fig, axes = plt.subplots(2, n + 1, figsize=(n + 1.5, 2.6))
    for row, (maps, name) in enumerate([(maps1, "conv1"), (maps2, "conv2")]):
        axes[row, 0].imshow(img.permute(1, 2, 0))
        axes[row, 0].set_title(name)
        for k in range(n):
            axes[row, k + 1].imshow(maps[0, k].cpu(), cmap="viridis")
        for ax in axes[row]:
            ax.axis("off")
    plt.tight_layout()
    plt.show()


for cls in ["River", "Highway", "Residential"]:
    i = d_test.targets.index(CLASSES.index(cls))
    img, _ = d_test[i]
    print(CLASSES_RU[CLASSES.index(cls)])
    show_feature_maps(img)

Карты `conv1` ещё очень похожи на исходный снимок. Каждая подсвечивает что-то простое: одна воду, другая светлые линии, третья зелень, четвёртая края объектов. Поищите на реке карту, где ярко горит только сама река.

Карты `conv2` вдвое меньше и заметно абстрактнее. Каждый нейрон второго слоя смотрит на участок побольше и реагирует уже не на отдельный перепад яркости, а на сочетания простых узоров. Чем глубже слой, тем меньше его карты похожи на снимок и тем больше они говорят о смысле: «здесь сетка улиц», «здесь вода с ровным берегом». А последний слой из этих признаков собирает ответ.

### 4.4 Куда смотрит сеть

Сеть уверенно говорит «река». Но на что она при этом смотрит, на саму реку или на поля вокруг неё? Проверить можно очень простым способом: будем по очереди закрывать разные участки снимка квадратом цвета фона и смотреть, насколько падает уверенность сети в правильном классе. Если закрыли что-то важное, уверенность сильно упадёт.

Так получается тепловая карта: чем ярче место, тем сильнее сеть на него опирается. Метод называется occlusion sensitivity.

In [ ]:
def occlusion_map(model, img, target, patch=16, stride=4):
    model.eval()
    positions = [(i, j) for i in range(0, 64 - patch + 1, stride)
                        for j in range(0, 64 - patch + 1, stride)]
    fill = img.mean(dim=(1, 2), keepdim=True)  # средний цвет снимка
    batch = img.unsqueeze(0).repeat(len(positions), 1, 1, 1)
    for k, (i, j) in enumerate(positions):
        batch[k, :, i:i + patch, j:j + patch] = fill

    with torch.no_grad():
        base = F.softmax(model(img.unsqueeze(0).to(device)), dim=1)[0, target].item()
        p = F.softmax(model(batch.to(device)), dim=1)[:, target].cpu()

    heat = torch.zeros(64, 64)
    count = torch.zeros(64, 64)
    for k, (i, j) in enumerate(positions):
        heat[i:i + patch, j:j + patch] += base - p[k]
        count[i:i + patch, j:j + patch] += 1
    return (heat / count).clamp(min=0)


show_classes = ["River", "Highway", "Residential", "Industrial", "SeaLake"]
fig, axes = plt.subplots(2, len(show_classes), figsize=(3 * len(show_classes), 6.4))
for j, cls in enumerate(show_classes):
    c = CLASSES.index(cls)
    # первый снимок этого класса, который сеть распознала правильно
    i = next(k for k in range(len(targets)) if targets[k] == c and preds[k] == c)
    heat = occlusion_map(best_model, best_test_ds[i][0], c)
    img = d_test[i][0].permute(1, 2, 0)

    axes[0, j].imshow(img)
    axes[0, j].set_title(CLASSES_RU[c])
    axes[1, j].imshow(img.mean(dim=2), cmap="gray")
    axes[1, j].imshow(heat, cmap="inferno", alpha=0.6)
    for ax in axes[:, j]:
        ax.axis("off")
axes[1, 0].set_title("куда смотрит сеть", loc="left")
plt.tight_layout()
plt.show()

Хорошо, если у реки подсвечена сама река, а у шоссе сама дорога. Но иногда сеть смотрит в неожиданные места, например на поля вокруг дороги, а не на дорогу. Это повод насторожиться: возможно, модель выучила не сам объект, а его типичное окружение. Такое называют shortcut learning. Это одна из самых частых проблем нейросетей, и её регулярно находят даже у больших моделей. Известный пример есть в статье про метод объяснения моделей LIME: классификатор, который «отличал» волков от хаски, на самом деле смотрел на снег на фоне.

Попробуйте поменять в коде классы или номера снимков и посмотреть на другие примеры, особенно на ошибки из 3.4.

### 4.5 Бонус: ваш район из космоса

Совсем необязательно, но интересно. Откройте Яндекс Карты или Google Maps в режиме «Спутник» и найдите свой район, дачу или любое другое место. Подберите масштаб так, чтобы в кадр помещался квадрат примерно 600 на 600 метров (ориентируйтесь по линейке масштаба в углу). Сделайте скриншот, вырежьте из него квадрат и сохраните рядом с ноутбуком как `my_place.png`. Ячейка ниже покажет, что думает о нём лучшая модель.

In [ ]:
path = "my_place.png"

if os.path.exists(path):
    img = Image.open(path).convert("RGB").resize((64, 64))
    best_model.eval()
    with torch.no_grad():
        p = F.softmax(best_model(best_transform(img).unsqueeze(0).to(device)), dim=1)[0].cpu()
    top = torch.argsort(p, descending=True)[:3].tolist()

    plt.imshow(img)
    plt.axis("off")
    plt.title(", ".join(f"{CLASSES_RU[i]} {p[i].item():.0%}" for i in top))
    plt.show()
else:
    print("Положите рядом с ноутбуком файл my_place.png")

Если сеть уверенно назвала ваш двор промзоной, не обижайтесь. Ваш снимок сильно отличается от тех, на которых она училась, и это называется сдвигом данных (domain shift):
- в EuroSAT снимки со спутника Sentinel-2 с разрешением 10 метров на пиксель. На картах обычно стоят гораздо более детальные снимки, часто аэрофотосъёмка, и после уменьшения до 64 на 64 они выглядят иначе;
- у разных спутников и камер разные цвета, а сеть к цветам очень чувствительна;
- в EuroSAT только Европа. Другая застройка, другие поля или снег сети незнакомы;
- у каждого снимка в датасете ровно один класс, а реальный квадрат 600 на 600 метров часто содержит всё сразу: дома, дорогу и кусок леса.

Для реальных проектов это обычная история. Модель, которая показывает 98% на тесте, в жизни может работать заметно хуже, если данные отличаются от обучающих. Поэтому модели всегда проверяют на данных, максимально похожих на те, с которыми им придётся работать.